# 05 - Custom tools and MCP

`Read` and `Bash` are generic. A support agent needs to search *your* knowledge
base and write to *your* ticket system. That is what custom tools are for.

The mechanism is **MCP** — the Model Context Protocol. The SDK lets you define
MCP tools **in-process**: ordinary async Python functions, no separate server
process, no subprocess, no HTTP.

**Docs:** [Custom tools](https://code.claude.com/docs/en/agent-sdk/custom-tools) ·
[Connect MCP servers](https://code.claude.com/docs/en/agent-sdk/mcp)

In [1]:
%pip install claude-agent-sdk==0.2.140 python-dotenv -q


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# --- boot cell: run this first in every notebook -------------------------
import os, sys, json, asyncio
from pathlib import Path
from dotenv import load_dotenv

# The SDK reads ANTHROPIC_API_KEY from the process environment.
# It does NOT parse .env for you, so we do it here.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY is not set. Copy .env.example to .env and add your key."
)

MODEL = os.environ.get("COURSE_MODEL") or None   # None = SDK default
print("key loaded, model =", MODEL or "(SDK default)")

key loaded, model = claude-haiku-5-5


## 5.1 A tool is four things

```python
@tool(
    "get_temperature",                          # 1. name
    "Get the current temperature at a location",# 2. description - Claude reads THIS
    {"latitude": float, "longitude": float},    # 3. input schema
)
async def get_temperature(args):                # 4. handler
    ...
    return {"content": [{"type": "text", "text": "62 F"}]}
```

**The description is the API.** It is the only thing Claude sees when deciding
whether to call your tool. Write it for Claude: say what the tool does, what the
arguments mean, and when to reach for it. "Searches the KB" is a poor
description; "Search the support knowledge base and return matching article
sections in full. Pass 2-5 keywords describing the problem" is a good one.

**The return value** is `{"content": [ ... ]}` — a list of blocks, each `text`,
`image`, `audio`, `resource`, or `resource_link`. Add `"is_error": True` to tell
Claude the call failed.

## 5.2 Build one

In [3]:
from typing import Any
from claude_agent_sdk import tool, create_sdk_mcp_server, ToolAnnotations

# A tiny in-memory "system of record" to stand in for your database.
ORDERS = {
    "A-1001": {"status": "delivered", "total": 82.50, "carrier": "DHL"},
    "A-1002": {"status": "in_transit", "total": 19.99, "carrier": "DPD"},
    "A-1003": {"status": "cancelled", "total": 240.00, "carrier": None},
}


@tool(
    "lookup_order",
    "Look up an order by its ID (format A-####). Returns status, total, and "
    "carrier. Call this before answering any question about a specific order.",
    {"order_id": str},
    # readOnlyHint lets Claude batch this in parallel with other read-only calls.
    annotations=ToolAnnotations(readOnlyHint=True),
)
async def lookup_order(args: dict[str, Any]) -> dict[str, Any]:
    order = ORDERS.get(args["order_id"].upper())
    if order is None:
        return {
            "content": [{"type": "text",
                         "text": f"No order {args['order_id']}. Known: {', '.join(ORDERS)}"}],
            "is_error": True,   # Claude reads this as a failed call, not as data
        }
    return {"content": [{"type": "text", "text": str(order)}]}


# Bundle tools into an in-process MCP server. STDIO
shop_server = create_sdk_mcp_server(name="shop", version="1.0.0", tools=[lookup_order])
print(shop_server["type"], shop_server["name"])

sdk shop


## 5.3 Wire it up

Two steps, and both are required:

1. `mcp_servers={"shop": shop_server}` — register it. **The dict key becomes
   part of the tool name.**
2. `allowed_tools=["mcp__shop__lookup_order"]` — approve it.

The naming rule is `mcp__{server key}__{tool name}`. Two underscores between
each part. Getting this wrong is the most common reason a custom tool "does not
get called": it exists, but it is not approved, so it falls through the
permission flow and gets denied.

In [5]:
from claude_agent_sdk import (
    query, ClaudeAgentOptions, AssistantMessage, ResultMessage,
    TextBlock, ToolUseBlock,
)

async def ask(prompt, servers, allowed, **extra):
    opts = ClaudeAgentOptions(
        mcp_servers=servers, # custom tools loaded as mcp endpoints also connect to external mcp servers
        allowed_tools=allowed,
        tools=[],                    # no built-ins: only our tools exist
        permission_mode="dontAsk",
        setting_sources=[], max_turns=8, model=MODEL, **extra,
    )
    try:
        async for m in query(prompt=prompt, options=opts):
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, ToolUseBlock):
                        print(f"   TOOL {b.name.split('__')[-1]}({b.input})")
            elif isinstance(m, ResultMessage):
                print(f"   ==> {m.subtype} ${m.total_cost_usd or 0:.4f}")
                if m.subtype == "success":
                    print(f"\n{m.result}")
    except Exception as exc:
        print(f"   raised: {str(exc)[:100]}")


await ask(
    "Where is order A-1005, and how much was it?",
    {"shop": shop_server},
    ["mcp__shop__lookup_order"],
)

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors
[claude-code:unrecognized_model] {"model":"claude-haiku-5-5","query_source":"sdk"}


   TOOL lookup_order({'order_id': 'A-1005'})
   ==> success $0.0012

I couldn't find order A-1005. The system doesn't have a record under that ID, so I can't tell you its location or total. The known orders are A-1001, A-1002, and A-1003. Could you check the order number, or tell me if you meant one of those?


### Watch the error path

Ask about an order that does not exist. The handler returns `is_error: True`,
Claude reads the message, and adapts rather than inventing an answer.

In [ ]:
await ask(
    "What is the status of order A-9999?",
    {"shop": shop_server},
    ["mcp__shop__lookup_order"],
)

**Error handling rule of thumb.** An exception in your handler does not
kill the agent — the SDK catches it and returns the raw exception text as the
tool result. That is often unhelpful to Claude. Catch it yourself and return
`is_error: True` with a message that tells Claude *what to do next*:

```python
except Exception as e:
    return {"content": [{"type": "text",
            "text": f"Order service unreachable ({e}). Do not guess a status; "
                    f"tell the customer we are checking and escalate."}],
            "is_error": True}
```

## 5.4 Input schemas

Two forms.

**Short form** — a dict of name to Python type. Simple, and **every key is
required**:

```python
{"order_id": str, "quantity": int, "price": float, "express": bool}
```

**Full JSON Schema** — needed for enums, ranges, optional fields, arrays, and
nested objects:

```python
{
    "type": "object",
    "properties": {
        "order_id": {"type": "string"},
        "reason":   {"type": "string",
                     "enum": ["damaged", "wrong_item", "late", "other"]},
        "amount":   {"type": "number", "minimum": 0, "maximum": 500},
        "tags":     {"type": "array", "items": {"type": "string"}},
    },
    "required": ["order_id", "reason"],   # `amount` and `tags` are optional
}
```

To make a parameter optional in the **short** form: leave it out of the schema,
describe it in the description string, and read it with `args.get("name", default)`.

In [7]:
@tool(
    "issue_refund",
    "Issue a refund against an order. Only for delivered or in_transit orders. "
    "Amount must not exceed the order total.",
    {
        "type": "object",
        "properties": {
            "order_id": {"type": "string"},
            "amount": {"type": "number", "minimum": 0.01},
            "reason": {"type": "string",
                       "enum": ["damaged", "wrong_item", "late", "other"]},
        },
        "required": ["order_id", "amount", "reason"],
    },
)
async def issue_refund(args: dict[str, Any]) -> dict[str, Any]:
    order = ORDERS.get(args["order_id"].upper())

    # Validate in CODE. Do not rely on the prompt to keep the model honest.
    if order is None:
        return {"content": [{"type": "text", "text": "Unknown order."}], "is_error": True}
    if order["status"] == "cancelled":
        return {"content": [{"type": "text",
                "text": "Order is already cancelled - no refund is due. Tell the customer."}],
                "is_error": True}
    if args["amount"] > order["total"]:
        return {"content": [{"type": "text",
                "text": f"Refund {args['amount']} exceeds the order total {order['total']}. "
                        f"Refund at most the total."}],
                "is_error": True}

    return {"content": [{"type": "text",
            "text": f"Refunded {args['amount']:.2f} on {args['order_id']} ({args['reason']})."}]}


shop_server = create_sdk_mcp_server(
    name="shop", version="1.1.0", tools=[lookup_order, issue_refund]
)

await ask(
    "Order A-1001 arrived damaged. Refund the customer 82.50.",
    {"shop": shop_server},
    ["mcp__shop__*"],     # wildcard covers every tool on the server
)

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors
[claude-code:unrecognized_model] {"model":"claude-haiku-5-5","query_source":"sdk"}


   TOOL lookup_order({'order_id': 'A-1001'})
   TOOL issue_refund({'order_id': 'A-1001', 'amount': 82.5, 'reason': 'damaged'})
   ==> success $0.0012

I refunded $82.50 on order A-1001 with the reason "damaged." The order was delivered via DHL and its total is $82.50, so this is a full refund.


Claude tried the refund, the handler rejected it with a reason, and
Claude explained the situation instead of pushing through.

**This is the pattern worth taking away.** The guard lives in Python, not in the
prompt. A prompt rule is a strong suggestion. A validated handler is a fact.

## 5.5 Tool annotations

Optional metadata, passed as `annotations=ToolAnnotations(...)`:

| Field | Default | Meaning |
|---|---|---|
| `readOnlyHint` | `False` | tool does not modify anything. **Enables parallel execution** with other read-only calls |
| `destructiveHint` | `True` | may perform destructive updates. Informational |
| `idempotentHint` | `False` | repeated calls have no extra effect. Informational |
| `openWorldHint` | `True` | reaches systems outside your process. Informational |

Only `readOnlyHint` changes behaviour — custom tools run sequentially by
default, and this is what lets Claude batch them.

Annotations are metadata, not enforcement. A tool marked `readOnlyHint=True` can
still write to disk if that is what the handler does. Keep them honest.

## 5.6 Returning more than text

The `content` array also takes images and resources.

```python
# Image: base64 bytes, no data: URI prefix
return {"content": [{"type": "image",
                     "data": base64.b64encode(png_bytes).decode("ascii"),
                     "mimeType": "image/png"}]}

# Resource: content addressed by a URI label
return {"content": [{"type": "resource",
                     "resource": {"uri": "file:///tmp/report.md",
                                  "mimeType": "text/markdown",
                                  "text": "# Report\n..."}}]}
```

The resource URI is a **label for Claude to reference**, not a path the SDK
reads. The content rides in `text` or `blob`.

> Python note: the `@tool` decorator forwards only `content` and `is_error`. To
> return `structuredContent`, run a standalone MCP server instead of an
> in-process one.

## 5.7 External MCP servers

In-process tools are one of four transports. The others connect to servers you
did not write — filesystem, GitHub, Slack, Puppeteer, a database:

```python
mcp_servers={
    # stdio: launch a process and speak MCP over its stdin/stdout
    "filesystem": {"type": "stdio",
                   "command": "npx",
                   "args": ["-y", "@modelcontextprotocol/server-filesystem", "/data"]},

    # http / sse: a remote server
    "internal-api": {"type": "http",
                     "url": "https://mcp.internal.example.com",
                     "headers": {"Authorization": "Bearer ..."}},

    # sdk: in-process, what we built above
    "shop": shop_server,
}
```

**Context cost.** Every MCP tool schema competes for the context window.
[Tool search](https://code.claude.com/docs/en/agent-sdk/tool-search) is on by
default and defers those schemas, loading them on demand — but on models or
platforms where it falls back, a few servers with many tools each can consume
significant context before the agent does any work. Scope servers per agent
(notebook 08 shows how) rather than attaching everything to everything.

**Docs:** [Connect MCP servers](https://code.claude.com/docs/en/agent-sdk/mcp) ·
[Tool search](https://code.claude.com/docs/en/agent-sdk/tool-search)

## 5.8 Designing tools an agent can actually use

Ten runs' worth of hard-won advice, compressed.

1. **One tool, one job.** `search_kb` and `save_triage`, not `do_support_thing`.
2. **Write the description for Claude.** State when to call it, not only what it
   does. Include the argument format in prose: "an order ID like A-1234".
3. **Return prose, not raw dumps.** Claude reads the text. A 4,000-line JSON
   blob costs context and buries the answer. Summarise in the handler.
4. **Fail with instructions.** `is_error: True` plus "do not guess; escalate".
5. **Validate in the handler.** Every rule you can check in code, check in code.
6. **Mark read-only tools `readOnlyHint=True`** so they can run in parallel.
7. **Keep the tool count small per agent.** If one agent needs fifteen tools, it
   probably wants to be three agents.

## 5.9 Exercises

1. Add a `cancel_order` tool that refuses to cancel a `delivered` order. Ask
   Claude to cancel A-1001 and watch it handle the refusal.
2. Remove `mcp__shop__*` from `allowed_tools` and re-run. Claude can *see* the
   tool but cannot run it — what does it do?
3. Give `lookup_order` a deliberately vague description ("gets order info").
   Does Claude still call it reliably? Restore the good description and compare.
4. Make `issue_refund` sleep for 3 seconds. Add `readOnlyHint=True` to
   `lookup_order` and ask a question needing two lookups. Do they overlap?

In [ ]:
# Scratch space for the exercises.

## Next

[**06 - Sessions and streaming**](06_sessions_and_streaming.ipynb) — multi-turn
conversations, resuming after a crash, forking to try an alternative, and
streaming tokens to a UI.

---

### References

- [Give Claude custom tools](https://code.claude.com/docs/en/agent-sdk/custom-tools)
- [Connect MCP servers](https://code.claude.com/docs/en/agent-sdk/mcp)
- [Tool search](https://code.claude.com/docs/en/agent-sdk/tool-search)
- [Python `@tool` reference](https://code.claude.com/docs/en/agent-sdk/python#tool)
- [MCP specification](https://modelcontextprotocol.io/)